In [1]:
#0-1　基本設定
import numpy as np
import pandas as pd
import pickle
import json
import os

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#0-2　テストデータからサンプル入力を生成

# FE2データを読込
test = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/加工データ/03FE2/test_FE2.zstd.parquet')

# モデルの特徴量名を取得
SAVE_DIR = "/content/drive/MyDrive/home_credit_models_gpu"
with open(os.path.join(SAVE_DIR, "lgbm_fold0.pkl"), "rb") as f:
    model = pickle.load(f)

feature_names = model.feature_name()
print(f"モデルの特徴量数: {len(feature_names)}")

モデルの特徴量数: 669


In [3]:
#1-1　サンプルリクエストの作成

# テストデータの1行目からサンプルを作成
sample_row = test.drop(columns=['SK_ID_CURR']).iloc[0]

# 特徴量辞書を作成（モデルの特徴量名に合わせる）
sample_features = {}
for name in feature_names:
    if name in sample_row.index:
        val = sample_row[name]
        # NaN/カテゴリ型の処理
        if pd.isna(val):
            sample_features[name] = 0
        elif isinstance(val, (np.integer, np.int64, np.int32, np.int16, np.int8)):
            sample_features[name] = int(val)
        elif isinstance(val, (np.floating, np.float64, np.float32)):
            sample_features[name] = float(val)
        else:
            sample_features[name] = 0  # カテゴリ型は0で代替
    else:
        sample_features[name] = 0

# JSON出力
sample_event = {"features": sample_features}
with open("sample_event.json", "w") as f:
    json.dump(sample_event, f, indent=2)

print(f"サンプルリクエスト作成完了: {len(sample_features)}特徴量")
print(f"ファイル: sample_event.json")

サンプルリクエスト作成完了: 669特徴量
ファイル: sample_event.json


In [4]:
#1-2　ローカルでの推論テスト

# モデルで直接推論して期待値を確認
input_array = np.array(
    [[sample_features.get(name, 0) for name in feature_names]],
    dtype=np.float64
)
prob = float(model.predict(input_array)[0])
threshold = 0.24
decision = "auto_approve" if prob < threshold else "manual_review"

print(f"\n=== ローカル推論テスト ===")
print(f"予測確率: {prob:.6f}")
print(f"閾値: {threshold}")
print(f"判定: {decision}")

#1-3　モデルファイルのサイズ確認

model_path = os.path.join(SAVE_DIR, "lgbm_fold0.pkl")
size_mb = os.path.getsize(model_path) / 1024**2
print(f"\nモデルファイルサイズ: {size_mb:.1f} MB")
print(f"※Lambdaの/tmpは512MBまで使用可能")


=== ローカル推論テスト ===
予測確率: 0.057771
閾値: 0.24
判定: auto_approve

モデルファイルサイズ: 8.8 MB
※Lambdaの/tmpは512MBまで使用可能


In [5]:
'''
=================================================================
AWSデプロイ手順（GUIベース）
=================================================================

■ 手順1: S3バケットの作成
  1. AWSコンソール → S3 → バケットを作成
  2. バケット名: home-credit-model-bucket（任意）
  3. リージョン: ap-northeast-1（大阪に近いリージョン）
  4. lgbm_fold0.pkl を models/ フォルダにアップロード

■ 手順2: ECRリポジトリの作成
  1. AWSコンソール → ECR → リポジトリを作成
  2. リポジトリ名: home-credit-predictor
  3. ローカルまたはCloud Shellで以下を実行:

     # ECRにログイン
     aws ecr get-login-password --region ap-northeast-1 | \
       docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.ap-northeast-1.amazonaws.com

     # Dockerイメージのビルド
     docker build -t home-credit-predictor .

     # タグ付けとプッシュ
     docker tag home-credit-predictor:latest \
       {ACCOUNT_ID}.dkr.ecr.ap-northeast-1.amazonaws.com/home-credit-predictor:latest
     docker push \
       {ACCOUNT_ID}.dkr.ecr.ap-northeast-1.amazonaws.com/home-credit-predictor:latest

■ 手順3: Lambda関数の作成
  1. AWSコンソール → Lambda → 関数の作成
  2. 「コンテナイメージ」を選択
  3. 関数名: home-credit-predictor
  4. コンテナイメージURI: 手順2でプッシュしたイメージを選択
  5. アーキテクチャ: x86_64
  6. 作成後に設定を変更:
     - メモリ: 1024MB（モデル読込に必要）
     - タイムアウト: 60秒（コールドスタート対策）
     - 環境変数:
       MODEL_BUCKET = home-credit-model-bucket
       MODEL_KEY = models/lgbm_fold0.pkl
       THRESHOLD = 0.24
  7. 実行ロールにS3読取権限（AmazonS3ReadOnlyAccess）を追加

■ 手順4: API Gatewayの作成
  1. AWSコンソール → API Gateway → REST APIを作成
  2. API名: home-credit-api
  3. リソース /predict を作成
  4. POSTメソッドを追加 → Lambda関数と統合
  5. APIをデプロイ（ステージ名: prod）
  6. エンドポイントURLが発行される

■ テスト方法
  curl -X POST \
    https://{api-id}.execute-api.ap-northeast-1.amazonaws.com/prod/predict \
    -H "Content-Type: application/json" \
    -d @sample_event.json

=================================================================
'''


'\n=================================================================\nAWSデプロイ手順（GUIベース）\n=================================================================\n \n■ 手順1: S3バケットの作成\n  1. AWSコンソール → S3 → バケットを作成\n  2. バケット名: home-credit-model-bucket（任意）\n  3. リージョン: ap-northeast-1（大阪に近いリージョン）\n  4. lgbm_fold0.pkl を models/ フォルダにアップロード\n \n■ 手順2: ECRリポジトリの作成\n  1. AWSコンソール → ECR → リポジトリを作成\n  2. リポジトリ名: home-credit-predictor\n  3. ローカルまたはCloud Shellで以下を実行:\n \n     # ECRにログイン\n     aws ecr get-login-password --region ap-northeast-1 |        docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.ap-northeast-1.amazonaws.com\n \n     # Dockerイメージのビルド\n     docker build -t home-credit-predictor .\n \n     # タグ付けとプッシュ\n     docker tag home-credit-predictor:latest        {ACCOUNT_ID}.dkr.ecr.ap-northeast-1.amazonaws.com/home-credit-predictor:latest\n     docker push        {ACCOUNT_ID}.dkr.ecr.ap-northeast-1.amazonaws.com/home-credit-predictor:latest\n \n■ 手順3: Lambda関数の作成\n  1. AWSコンソー